<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B01%5D%20-%20Intro_No_Supervisados/%5B01%5D%20-%20Notebooks/E4_Defiende_tu_K.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E4 · Defiende tu K - Introducción a los modelos no supervisados (bonus)

## Introducción

Elegir K no es solo mirar una métrica: es una **decisión** que tienes que **defender**. En
este ejercicio eliges el K final y lo justificas con **3 evidencias**:

1. **Cuantitativa**: método del codo y Silhouette Score.
2. **Interpretabilidad**: ¿los grupos se entienden y tienen tamaños razonables?
3. **Acción de negocio**: ¿cada grupo sugiere una decisión distinta?

El resultado es una **recomendación ejecutiva**.

## Objetivos del ejercicio

- Reunir evidencia cuantitativa (codo + silhouette) para varios K.
- Valorar la **interpretabilidad** (tamaños y diferencias entre grupos).
- Conectar el K con una **acción de negocio**.
- Escribir una **recomendación** que defienda tu elección.

## Descripción del dataset (clientes sin etiqueta)

Imagina que tienes una base de clientes y quieres ofrecerles un descuento, pero **no hay
etiquetas**: nadie te ha dicho qué cliente es de qué tipo. El objetivo del aprendizaje no
supervisado es justo ese: **descubrir la estructura** que hay dentro de los datos.

Generamos un dataset **sintético y reproducible** con `generar_clientes` (autocontenido en
Colab). Cada fila es un cliente con estas variables:

| Variable | Tipo | Descripción |
|---|---|---|
| `gasto_anual` | numérica | Gasto total al año (€), escala de miles |
| `num_visitas` | numérica | Nº de visitas al año, escala de decenas |
| `ticket_medio` | numérica | Gasto medio por compra (€) |
| `antiguedad_meses` | numérica | Meses como cliente |
| `edad` | numérica | Edad del cliente |
| `usa_app` | binaria | 1 si usa la app |
| `tiene_tarjeta_fidelidad` | binaria | 1 si tiene tarjeta de fidelidad |
| `compra_online` | binaria | 1 si compra online |
| `recibe_newsletter` | binaria | 1 si recibe la newsletter |
| `devuelve_productos` | binaria | 1 si suele devolver productos |

> Fíjate en las **escalas tan distintas** (gasto en miles, visitas en decenas). Esto va a ser
> clave: en clustering, la distancia depende de la escala, así que **habrá que escalar**.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

### 2. Datos y escalado

In [ ]:
import numpy as np
import pandas as pd

def generar_clientes(n=600, semilla=42):
    # Dataset sintetico y reproducible de clientes SIN ETIQUETA para segmentacion.
    # Por dentro hay 4 perfiles latentes que el modelo deberia redescubrir, pero NO los
    # exponemos: en aprendizaje no supervisado no hay target, solo buscamos estructura.
    rng = np.random.default_rng(semilla)
    # perfil: (gasto_anual, num_visitas, ticket_medio, antiguedad_meses, edad,
    #          p_app, p_fidelidad, p_online, p_newsletter, p_devuelve)
    perfiles = [
        (9000, 42, 230, 60, 46, 0.85, 0.90, 0.70, 0.60, 0.10),  # grandes clientes
        (1100,  6, 120, 22, 37, 0.40, 0.20, 0.55, 0.30, 0.10),  # ocasionales
        (3200, 36,  75, 44, 52, 0.50, 0.65, 0.60, 0.80, 0.55),  # cazaofertas
        (2400, 15, 165,  9, 30, 0.92, 0.40, 0.95, 0.50, 0.20),  # nuevos digitales
    ]
    pesos = [0.22, 0.33, 0.25, 0.20]
    seg = rng.choice(len(perfiles), size=n, p=pesos)

    filas = []
    for s in seg:
        g, v, t, a, e, pa, pf, po, pn, pdv = perfiles[s]
        filas.append([
            round(max(50, rng.normal(g, g * 0.22)), 2),    # gasto_anual (€)
            int(max(1, round(rng.normal(v, v * 0.30)))),    # num_visitas
            round(max(5, rng.normal(t, t * 0.22)), 2),      # ticket_medio (€)
            int(max(1, round(rng.normal(a, 12)))),          # antiguedad_meses
            int(np.clip(rng.normal(e, 8), 18, 85)),         # edad
            int(rng.random() < pa),                          # usa_app
            int(rng.random() < pf),                          # tiene_tarjeta_fidelidad
            int(rng.random() < po),                          # compra_online
            int(rng.random() < pn),                          # recibe_newsletter
            int(rng.random() < pdv),                         # devuelve_productos
        ])
    cols = ["gasto_anual", "num_visitas", "ticket_medio", "antiguedad_meses", "edad",
            "usa_app", "tiene_tarjeta_fidelidad", "compra_online", "recibe_newsletter",
            "devuelve_productos"]
    return pd.DataFrame(filas, columns=cols)

In [ ]:
df = generar_clientes(n=600, semilla=42)
X_esc = StandardScaler().fit_transform(df.to_numpy(dtype=float))

### 3. Evidencia 1: codo y silhouette (K = 2..8)

In [ ]:
Ks = range(2, 9)
inercias, sils = [], []
for k in Ks:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=0).fit(X_esc)
    inercias.append(km.inertia_)
    sils.append(silhouette_score(X_esc, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(list(Ks), inercias, marker="o"); ax[0].set_title("Codo (inercia)")
ax[0].set_xlabel("K"); ax[0].set_ylabel("Inercia")
ax[1].plot(list(Ks), sils, marker="o", color="#27ae60"); ax[1].set_title("Silhouette")
ax[1].set_xlabel("K"); ax[1].set_ylabel("Silhouette")
plt.tight_layout(); plt.show()

for k, s in zip(Ks, sils):
    print(f"K={k}: silhouette={s:.3f}")

### 4. Evidencia 2: interpretabilidad (tamaños de los grupos)

In [ ]:
print("Reparto de clientes por cluster para cada K candidato:")
for k in [2, 3, 4, 5]:
    lab = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=0).fit_predict(X_esc)
    tam = np.bincount(lab)
    print(f"  K={k}: tamaños {tam}  (mínimo {tam.min()} clientes)")
print("\nGrupos demasiado pequeños suelen ser poco accionables.")

### 5. Evidencia 3: acción de negocio (¿perfiles distintos?)

In [ ]:
# Para el K candidato, ¿los grupos se diferencian en variables de negocio?
k_cand = int(list(Ks)[int(np.argmax(sils))])
lab = KMeans(n_clusters=k_cand, init="k-means++", n_init=10, random_state=0).fit_predict(X_esc)
perfil = df.assign(cluster=lab).groupby("cluster")[["gasto_anual", "num_visitas", "ticket_medio"]].mean().round(0)
print(f"Perfil de negocio con K={k_cand}:")
print(perfil)
print("\nSi cada grupo pide una campaña distinta, el K es útil para el negocio.")

### 6. Recomendación ejecutiva

Junta las tres evidencias y redacta tu recomendación. Un ejemplo de cómo se vería (ajústalo a
tus números):

In [ ]:
k_final = k_cand
print("=" * 60)
print(f"RECOMENDACIÓN: usar K = {k_final}")
print("=" * 60)
print(f"1) Cuantitativa: silhouette máximo en K={k_final} y el codo se estabiliza ahí.")
print(f"2) Interpretabilidad: {k_final} grupos de tamaño razonable y describibles.")
print(f"3) Negocio: cada grupo sugiere una campaña distinta (ver perfil).")
print("\nConclusión: K =", k_final, "equilibra métrica, claridad y utilidad de negocio.")

### Ahora tú


Partiendo del siguiente dataset real, realiza un análisis clúster, crea los perfiles y propón una recomendación ejecutiva como hemos visto anteriormente.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/dtoralg/TheValley_MDS/refs/heads/main/%5B01%5D%20-%20Intro_No_Supervisados/%5B00%5D%20-%20Data/credit_card_data.csv")
df.head()

In [ ]:
# Tu codigo aquí

In [ ]:
# Recomendación ejecutiva
print("=" * 60)
print(f"RECOMENDACIÓN: usar K = xx")
print("=" * 60)
print(f"1) Cuantitativa: xx")
print(f"2) Interpretabilidad: xx")
print(f"3) Negocio: xx")
print("\nConclusión: K =", k_final, "por xx")

### Reflexión

1. ¿Qué harías si la métrica (silhouette) recomienda un K poco útil para el negocio?
2. ¿Por qué un K con grupos diminutos puede ser mala idea aunque la métrica suba?
3. ¿Cómo defenderías tu K ante alguien que no es técnico?
4. ¿Qué evidencia pesa más en tu decisión y por qué?